In [ ]:
import os
import json
import glob
import re
import numpy as np
import pandas as pd
from typing import Dict, Any, List, Tuple

# Determine base directory safely in both scripts and Jupyter notebooks
try:
    _BASE_DIR = os.path.dirname(__file__)
except NameError:
    _BASE_DIR = os.getcwd()
RUNS_DIR = os.path.join(_BASE_DIR, '.')

def _safe_get_metrics(m: Dict[str, Any]) -> Tuple[float, float, float]:
    # Handle possible key variants
    acc = float(m.get('accuracy', np.nan))
    bacc = float(m.get('balanced_accuracy', m.get('balanced_acc', np.nan)))
    auc = float(m.get('auc', np.nan))
    return acc, bacc, auc

def parse_metrics_ensemble(exp_dir: str) -> Dict[str, Any]:
    f = os.path.join(exp_dir, 'metrics_ensemble.json')
    if not os.path.exists(f):
        return {}
    with open(f, 'r') as fh:
        data = json.load(fh)
    acc, bacc, auc = _safe_get_metrics(data.get('metrics', {}))
    return {
        'ensemble_accuracy': acc,
        'ensemble_balanced_accuracy': bacc,
        'ensemble_auc': auc,
        'counts': data.get('counts', {}),
        'timing': data.get('timing', {}),
        'num_classes': data.get('num_classes', None),
        'data_flag': data.get('data_flag', None),
    }

def parse_fold_metrics(exp_dir: str) -> Dict[str, Any]:
    files = sorted(glob.glob(os.path.join(exp_dir, 'metrics_fold_*_test.json')))
    accs, baccs, aucs = [], [], []
    n_samples = None
    for f in files:
        try:
            with open(f, 'r') as fh:
                d = json.load(fh)
            acc, bacc, auc = _safe_get_metrics(d.get('metrics', {}))
            if not np.isnan(acc): accs.append(acc)
            if not np.isnan(bacc): baccs.append(bacc)
            if not np.isnan(auc): aucs.append(auc)
            if n_samples is None and isinstance(d.get('counts', {}), dict):
                n_samples = d['counts'].get('n_samples', None)
        except Exception:
            continue
    res = {
        'folds_found': len(files),
        'fold_acc_std': float(np.std(accs, ddof=1)) if len(accs) >= 2 else np.nan,
        'fold_balanced_accuracy_std': float(np.std(baccs, ddof=1)) if len(baccs) >= 2 else np.nan,
        'fold_auc_std': float(np.std(aucs, ddof=1)) if len(aucs) >= 2 else np.nan,
        'fold_acc_values': accs,
        'fold_bacc_values': baccs,
        'fold_auc_values': aucs,
        'n_samples': n_samples,
    }
    return res

_log_epoch_re = re.compile(r'^fold_(\d+)\s+epoch=(\d+).*?epoch_time_s=([0-9.]+)')
_total_train_re = re.compile(r'^fold_(\d+)\s+total_train_sec=([0-9.]+)')
_best_epoch_re = re.compile(r'fold_(\d+).*?best_epoch(?:_idx)?=(\d+)', re.IGNORECASE)


def parse_metrics_log(exp_dir: str) -> Dict[str, Any]:
    f = os.path.join(exp_dir, 'metrics.log')
    if not os.path.exists(f):
        return {}
    fold_max_epoch = {}
    all_epoch_times = []
    train_secs_by_fold: Dict[int, float] = {}
    best_epoch_by_fold: Dict[int, int] = {}
    try:
        with open(f, 'r') as fh:
            for line in fh:
                s = line.strip()
                m = _log_epoch_re.search(s)
                if m:
                    fold = int(m.group(1))
                    epoch = int(m.group(2))
                    etime = float(m.group(3))
                    fold_max_epoch[fold] = max(epoch, fold_max_epoch.get(fold, -1))
                    all_epoch_times.append(etime)
                    # do not continue; a line could contain both epoch and best_epoch
                b = _best_epoch_re.search(s)
                if b:
                    fold_b = int(b.group(1))
                    best_e = int(b.group(2))
                    best_epoch_by_fold[fold_b] = best_e
                t = _total_train_re.search(s)
                if t:
                    fold_t = int(t.group(1))
                    train_secs_by_fold[fold_t] = float(t.group(2))
    except Exception:
        pass

    # Convert indexes to counts (assume 0-based in logs; if 1-based, remove +1)
    trained_counts = [v + 1 for v in sorted(fold_max_epoch.values())]
    kept_counts = [v + 1 for v in sorted(best_epoch_by_fold.values())]

    # Preferred: kept (best) epochs; fallback: trained counts
    if kept_counts:
        mean_epochs = float(np.mean(kept_counts))
        std_epochs = float(np.std(kept_counts, ddof=1)) if len(kept_counts) >= 2 else np.nan
    else:
        mean_epochs = float(np.mean(trained_counts)) if trained_counts else np.nan
        std_epochs = float(np.std(trained_counts, ddof=1)) if len(trained_counts) >= 2 else np.nan

    mean_time_per_epoch = float(np.mean(all_epoch_times)) if all_epoch_times else np.nan
    fold_train_secs = [train_secs_by_fold[k] for k in sorted(train_secs_by_fold.keys())]
    mean_train_sec = float(np.mean(fold_train_secs)) if fold_train_secs else np.nan
    std_train_sec = float(np.std(fold_train_secs, ddof=1)) if len(fold_train_secs) >= 2 else np.nan

    return {
        'fold_epoch_counts': trained_counts,
        'best_epoch_counts': kept_counts,
        'mean_fold_nb_epochs': mean_epochs,                 # used by table (kept if available)
        'std_fold_nb_epochs': std_epochs,                   # used by table (kept if available)
        'mean_fold_nb_epochs_kept': float(np.mean(kept_counts)) if kept_counts else np.nan,
        'std_fold_nb_epochs_kept': float(np.std(kept_counts, ddof=1)) if len(kept_counts) >= 2 else (np.nan if kept_counts else np.nan),
        'mean_fold_nb_epochs_trained': float(np.mean(trained_counts)) if trained_counts else np.nan,
        'std_fold_nb_epochs_trained': float(np.std(trained_counts, ddof=1)) if len(trained_counts) >= 2 else (np.nan if trained_counts else np.nan),
        'mean_time_per_epoch_s': mean_time_per_epoch,
        'n_epoch_lines': len(all_epoch_times),
        'fold_train_secs': fold_train_secs,
        'mean_fold_train_sec': mean_train_sec,
        'std_fold_train_sec': std_train_sec,
    }

# Parse augmentation flag and batch size from experiment folder name
_config_re = re.compile(r'randaug(\d+).*_bs(\d+)', re.IGNORECASE)

def parse_exp_config(exp_dir_name: str) -> Dict[str, Any]:
    m = _config_re.search(exp_dir_name)
    if not m:
        return {'aug_label': '', 'batch_size': None}
    aug = int(m.group(1))
    bs = int(m.group(2))
    return {'aug_label': ('aug' if aug else 'no aug'), 'batch_size': bs}

def list_last_n_experiments(dataset_dir: str, n: int = 2) -> List[str]:
    exps = [os.path.join(dataset_dir, d) for d in os.listdir(dataset_dir)
            if os.path.isdir(os.path.join(dataset_dir, d))]
    # Prefer sorting by metrics.log mtime, fallback to dir mtime
    def sort_key(p):
        logf = os.path.join(p, 'metrics.log')
        try:
            return os.path.getmtime(logf)
        except Exception:
            return os.path.getmtime(p)
    exps.sort(key=sort_key)  # ascending
    return exps[-n:] if len(exps) >= n else exps

def parse_dataset(dataset_dir: str, last_n: int = 2) -> Dict[str, Any]:
    dataset = os.path.basename(dataset_dir.rstrip('/'))
    results = {
        'dataset': dataset,
        'experiments': []
    }
    for exp in list_last_n_experiments(dataset_dir, last_n):
        exp_name = os.path.basename(exp)
        ens = parse_metrics_ensemble(exp)
        folds = parse_fold_metrics(exp)
        logm = parse_metrics_log(exp)
        cfg = parse_exp_config(exp_name)
        results['experiments'].append({
            'exp_dir': exp_name,
            **cfg,
            **ens,
            **folds,
            **logm
        })
    return results

def _pretty_dataset_name(ds: str) -> str:
    mapping = {
        'pathmnist': 'Path',
        'dermamnist': 'Derma',
        'dermamnist-e': 'Derma-e',
        # split over two lines in the multirow cell
        'dermamnist-e external': r'\makecell{Derma-e\\external}',
        'octmnist': 'OCT',
        'pneumoniamnist': 'Pneumonia',
        'breastmnist': 'Breast',
        'bloodmnist': 'Blood',
        'tissuemnist': 'Tissue',
        'organamnist': 'Organ',
        'organcmnist': 'Organ',
        'organmnist': 'Organ',
    }
    return mapping.get(ds.lower(), ds.capitalize())

def _fmt_mean_std(mean: float, std: float, decimals: int = 3) -> str:
    if mean is None or np.isnan(mean):
        return ''
    if std is None or np.isnan(std):
        return f'{mean:.{decimals}f}'
    return f'{mean:.{decimals}f}({std:.{decimals}f})'

def _fmt_pm(mean: float, std: float, decimals: int = 1) -> str:
    if mean is None or np.isnan(mean):
        return ''
    if std is None or np.isnan(std):
        return f'{mean:.{decimals}f}'
    return f'{mean:.{decimals}f} ± {std:.{decimals}f}'

def build_latex_table(rows: List[Dict[str, Any]]) -> str:
    # Group by dataset, then order aug rows as ['no aug', 'aug']
    from collections import defaultdict
    by_ds = defaultdict(list)
    for r in rows:
        by_ds[r['dataset']].append(r)
    for ds in by_ds:
        by_ds[ds].sort(key=lambda x: (0 if x.get('aug_label') == 'no aug' else 1, x.get('experiment','')))
    lines = []
    header = [
        r'\begin{table}[htb]',
        r'     \small',
        r'     \caption{ResNet-18 results on MedMNIST datasets (Ensemble metrics with SD across folds; mean kept epochs $\pm$ SD; mean time/epoch).}',
        r'     \centering',
        r'     \begin{tabular}{l*{7}{c}}',
        r'          \toprule',
        r'          \multirow{2}{*}{Dataset} & \multirow{2}{*}{} & \multirow{2}{*}{\makecell{Batch \\ Size}} & \multirow{2}{*}{Acc} & \multirow{2}{*}{b-Acc} & \multirow{2}{*}{AUC} & \multirow{2}{*}{\makecell{Mean nb \\ epochs (kept)}} & \multirow{2}{*}{\makecell{Mean \\ time/epoch(s)}}\\ [12pt]',
        r'          \midrule',
        ''
    ]
    lines.extend(header)
    for ds in sorted(by_ds.keys()):
        exps = by_ds[ds]
        n = len(exps)
        if n == 0:
            continue
        for i, r in enumerate(exps):
            ds_cell = (r'\multirow{' + str(n) + r'}{*}{' + _pretty_dataset_name(ds) + r'}') if i == 0 else ''
            aug = r.get('aug_label', '')
            bs = r.get('batch_size', '')
            acc = _fmt_mean_std(r.get('ensemble_acc'), r.get('fold_acc_std'), 3)
            bacc = _fmt_mean_std(r.get('ensemble_bal_acc'), r.get('fold_bal_acc_std'), 3)
            auc = _fmt_mean_std(r.get('ensemble_auc'), r.get('fold_auc_std'), 3)
            e_mean = r.get('mean_fold_nb_epochs')
            e_std = r.get('std_fold_nb_epochs')
            epochs_cell = _fmt_pm(e_mean, e_std, 1)
            t_per_epoch = r.get('mean_time_per_epoch_s')
            tcell = (r'\num{' + f'{t_per_epoch:.1f}' + r'}') if (t_per_epoch is not None and not np.isnan(t_per_epoch)) else ''
            row_end = r'\\[8pt]' if i == n - 1 else r'\\'
            line = f'          {ds_cell} & {aug} & {bs} & {acc} & {bacc} & {auc} & {epochs_cell} & {tcell} {row_end}'
            lines.append(line)
    lines.append(r'          \bottomrule')
    lines.append(r'     \end{tabular}')
    lines.append(r'\end{table}')
    return '\n'.join(lines)

def _append_derma_external_rows(rows: List[Dict[str, Any]]) -> None:
    # No aug (external)
    rows.append({
        'dataset': 'dermamnist-e external',
        'experiment': 'external_no_aug',
        'aug_label': 'no aug',
        'batch_size': '',  # blank cell
        'ensemble_acc': 0.5981,
        'ensemble_bal_acc': 0.6109,
        'ensemble_auc': 0.8839,
        'fold_acc_std': 0.0265,
        'fold_bal_acc_std': 0.0788,
        'fold_auc_std': 0.0164,
        'mean_fold_nb_epochs': None,     # blank cell via formatter
        'std_fold_nb_epochs': None,      # blank cell via formatter
        'mean_time_per_epoch_s': None,   # blank cell
    })
    # Aug (external)
    rows.append({
        'dataset': 'dermamnist-e external',
        'experiment': 'external_aug',
        'aug_label': 'aug',
        'batch_size': '',  # blank cell
        'ensemble_acc': 0.5665,
        'ensemble_bal_acc': 0.6315,
        'ensemble_auc': 0.9005,
        'fold_acc_std': 0.0467,
        'fold_bal_acc_std': 0.0182,
        'fold_auc_std': 0.0116,
        'mean_fold_nb_epochs': None,
        'std_fold_nb_epochs': None,
        'mean_time_per_epoch_s': None,
    })
    
def main(runs_dir: str = RUNS_DIR, last_n: int = 2,
         out_json: str = 'summary_last2.json',
         out_csv: str = 'summary_last2.csv',
         out_tex: str = 'summary_last2.tex'):
    datasets = [os.path.join(runs_dir, d) for d in os.listdir(runs_dir)
                if os.path.isdir(os.path.join(runs_dir, d))]
    summaries = []
    rows = []
    for ds_dir in sorted(datasets):
        # Skip only non-datasets
        ds_name = os.path.basename(ds_dir.rstrip('/')).lower()
        if ds_name in ('.ipynb_checkpoints',):
            continue
        res = parse_dataset(ds_dir, last_n=last_n)
        summaries.append(res)
        for e in res['experiments']:
            rows.append({
                'dataset': res['dataset'],
                'experiment': e.get('exp_dir'),
                'aug_label': e.get('aug_label'),
                'batch_size': e.get('batch_size'),
                'ensemble_acc': e.get('ensemble_accuracy'),
                'ensemble_bal_acc': e.get('ensemble_balanced_accuracy'),
                'ensemble_auc': e.get('ensemble_auc'),
                'folds_found': e.get('folds_found'),
                'fold_acc_std': e.get('fold_acc_std'),
                'fold_bal_acc_std': e.get('fold_balanced_accuracy_std'),
                'fold_auc_std': e.get('fold_auc_std'),
                'mean_fold_nb_epochs': e.get('mean_fold_nb_epochs'),
                'std_fold_nb_epochs': e.get('std_fold_nb_epochs'),
                'mean_time_per_epoch_s': e.get('mean_time_per_epoch_s'),
                'mean_fold_train_sec': e.get('mean_fold_train_sec'),
                'std_fold_train_sec': e.get('std_fold_train_sec'),
            })

    # Inject the requested external rows
    _append_derma_external_rows(rows)

    out_json_path = os.path.join(runs_dir, out_json)
    with open(out_json_path, 'w') as fh:
        json.dump(summaries, fh, indent=2)
    out_csv_path = os.path.join(runs_dir, out_csv)
    df = pd.DataFrame(rows)
    df.to_csv(out_csv_path, index=False)
    latex = build_latex_table(rows)
    out_tex_path = os.path.join(runs_dir, out_tex)
    with open(out_tex_path, 'w') as fh:
        fh.write(latex)
    print(f"Saved summary JSON to: {out_json_path}")
    print(f"Saved summary CSV to:  {out_csv_path}")
    print(f"Saved LaTeX table to:  {out_tex_path}")
    if not df.empty:
        print("\nLaTeX table preview:\n")
        print(latex)

if __name__ == '__main__':
    main()

# Generate LaTeX Tables from Parsed Results

This section generates the requested LaTeX tables with ensemble results + std across folds.

In [ ]:
import os
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

# Configuration for LaTeX table generation
RUNS_DIR = Path('.')

# Dataset ordering for tables
DATASETS_TABLE1 = ['pathmnist', 'dermamnist-e', 'dermamnist-e_external', 'octmnist', 'pneumoniamnist']
DATASETS_TABLE2 = ['breastmnist', 'bloodmnist', 'tissuemnist', 'organamnist', 'amos22']

# Dataset name mapping
DATASET_NAMES = {
    'pathmnist': 'Path',
    'dermamnist-e': 'Derma-e',
    'dermamnist-e_external': r'\makecell{Derma-e\\ext.}',
    'octmnist': 'OCT',
    'pneumoniamnist': 'Pneum.',
    'breastmnist': 'Breast',
    'bloodmnist': 'Blood',
    'tissuemnist': 'Tissue',
    'organamnist': 'Organ',
    'amos22': 'AMOS',
}

# Parse experiment directory name to extract model type and setup
# Fixed pattern: dropout is at the end with format dropout0.1 or dropout0.3
EXP_DIR_PATTERN = re.compile(
    r'(?P<model>resnet18|vit_b_16)_224_.*?randaug(?P<randaug>[01]).*?(?:dropout(?P<dropout>\d\.\d+))?$'
)

def parse_experiment_config(exp_name):
    """Extract model type, augmentation, and dropout from experiment name."""
    match = EXP_DIR_PATTERN.search(exp_name)
    if not match:
        return None
    
    model = 'resnet18' if 'resnet18' in match.group('model') else 'vit_b_16'
    has_aug = match.group('randaug') == '1'
    has_dropout = match.group('dropout') is not None
    
    # Determine setup
    if has_aug and has_dropout:
        setup = 'DADO'
    elif has_aug:
        setup = 'DA'
    elif has_dropout:
        setup = 'DO'
    else:
        setup = '--'
    
    return {
        'model': model,
        'setup': setup,
        'has_aug': has_aug,
        'has_dropout': has_dropout,
        'dropout_value': match.group('dropout')
    }

def load_experiment_metrics(exp_dir):
    """Load ensemble and fold metrics from an experiment directory."""
    exp_path = Path(exp_dir)
    
    # Load ensemble metrics
    ensemble_file = exp_path / 'metrics_ensemble.json'
    if not ensemble_file.exists():
        return None
    
    with open(ensemble_file, 'r') as f:
        ensemble_data = json.load(f)
    
    ensemble_metrics = ensemble_data.get('metrics', {})
    
    # Load fold metrics - try both naming patterns
    fold_metrics = []
    
    # First try: metrics_fold_*_test.json (ResNet pattern)
    fold_files = sorted(exp_path.glob('metrics_fold_*_test.json'))
    
    # Second try: metrics_fold_*.json (ViT pattern) - but exclude metrics_ensemble.json
    if not fold_files:
        fold_files = sorted([f for f in exp_path.glob('metrics_fold_*.json') 
                            if f.name != 'metrics_ensemble.json'])
    
    for fold_file in fold_files:
        with open(fold_file, 'r') as f:
            fold_data = json.load(f)
            fold_metrics.append(fold_data.get('metrics', {}))
    
    if not fold_metrics:
        return None
    
    # Extract balanced accuracy and AUC
    ensemble_bacc = ensemble_metrics.get('balanced_accuracy', ensemble_metrics.get('balanced_acc', np.nan))
    ensemble_auc = ensemble_metrics.get('auc', np.nan)
    
    fold_baccs = [m.get('balanced_accuracy', m.get('balanced_acc', np.nan)) for m in fold_metrics]
    fold_aucs = [m.get('auc', np.nan) for m in fold_metrics]
    
    # Filter out NaNs
    fold_baccs = [x for x in fold_baccs if not np.isnan(x)]
    fold_aucs = [x for x in fold_aucs if not np.isnan(x)]
    
    return {
        'ensemble_bacc': ensemble_bacc,
        'ensemble_auc': ensemble_auc,
        'std_bacc': np.std(fold_baccs, ddof=1) if len(fold_baccs) > 1 else 0.0,
        'std_auc': np.std(fold_aucs, ddof=1) if len(fold_aucs) > 1 else 0.0,
        'num_folds': len(fold_metrics)
    }

def collect_all_results():
    """Collect all results from dataset directories."""
    results = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
    
    # Iterate through dataset directories
    for dataset_dir in RUNS_DIR.iterdir():
        if not dataset_dir.is_dir():
            continue
        
        dataset_name = dataset_dir.name.lower()
        
        # Skip non-dataset directories
        if dataset_name in ['.ipynb_checkpoints', 'clean_models', '__pycache__']:
            continue
        
        print(f"Processing {dataset_name}...")
        
        # Find all experiment directories
        exp_dirs_found = 0
        for exp_dir in dataset_dir.iterdir():
            if not exp_dir.is_dir():
                continue
            
            exp_dirs_found += 1
            exp_name = exp_dir.name
            
            # Debug: Show all directory names being checked
            if 'vit' in exp_name.lower():
                print(f"  [DEBUG] Found ViT directory: {exp_name}")
            
            # Parse experiment configuration
            config = parse_experiment_config(exp_name)
            if not config:
                if 'vit' in exp_name.lower():
                    print(f"  [DEBUG] ✗ Parse FAILED for ViT: {exp_name}")
                continue
            
            if 'vit' in exp_name.lower():
                print(f"  [DEBUG] ✓ Parse OK for ViT: {config}")
            
            # Load metrics
            metrics = load_experiment_metrics(exp_dir)
            if not metrics:
                if 'vit' in exp_name.lower():
                    print(f"  [DEBUG] ✗ Metrics load FAILED for ViT: {exp_name}")
                    # Check if metrics files exist
                    ensemble_file = exp_dir / 'metrics_ensemble.json'
                    print(f"  [DEBUG]   metrics_ensemble.json exists: {ensemble_file.exists()}")
                    fold_files = list(exp_dir.glob('metrics_fold_*_test.json'))
                    print(f"  [DEBUG]   Found {len(fold_files)} fold files")
                continue
            
            if 'vit' in exp_name.lower():
                print(f"  [DEBUG] ✓ Metrics loaded for ViT: {metrics}")
            
            model = config['model']
            setup = config['setup']
            
            # Store results (keep the most recent if duplicate)
            results[dataset_name][model][setup] = metrics
            
            if 'vit' in exp_name.lower():
                print(f"  [DEBUG] ✓ Stored ViT result: dataset={dataset_name}, model={model}, setup={setup}")
            
            print(f"  {model} {setup}: bAcc={metrics['ensemble_bacc']:.3f}±{metrics['std_bacc']:.3f}, "
                  f"AUC={metrics['ensemble_auc']:.3f}±{metrics['std_auc']:.3f}")
        
        print(f"  Total experiment directories found: {exp_dirs_found}")
    
    return results

def format_cell(mean, std):
    """Format table cell as mean±std."""
    if np.isnan(mean) or np.isnan(std):
        return ''
    return f"{mean:.3f}$\\pm${std:.3f}"

def generate_latex_table_rows(results, datasets):
    """Generate LaTeX table rows for specified datasets."""
    rows = []
    
    for model_type, model_key in [('ResNet18', 'resnet18'), ('ViT', 'vit_b_16')]:
        model_rows = []
        
        for setup in ['--', 'DA', 'DO', 'DADO']:
            row_parts = ['', setup]  # Empty first column for multirow, then setup
            
            for dataset in datasets:
                # Handle special cases for external datasets
                if dataset == 'dermamnist-e_external':
                    # Use dermamnist-e models evaluated on external set
                    source_dataset = 'dermamnist-e'
                elif dataset == 'amos22':
                    # Use organamnist models evaluated on AMOS
                    source_dataset = 'organamnist'
                else:
                    source_dataset = dataset
                
                if (dataset in results and 
                    model_key in results[dataset] and 
                    setup in results[dataset][model_key]):
                    metrics = results[dataset][model_key][setup]
                    bacc_cell = format_cell(metrics['ensemble_bacc'], metrics['std_bacc'])
                    auc_cell = format_cell(metrics['ensemble_auc'], metrics['std_auc'])
                elif (source_dataset in results and 
                      model_key in results[source_dataset] and 
                      setup in results[source_dataset][model_key]):
                    metrics = results[source_dataset][model_key][setup]
                    bacc_cell = format_cell(metrics['ensemble_bacc'], metrics['std_bacc'])
                    auc_cell = format_cell(metrics['ensemble_auc'], metrics['std_auc'])
                else:
                    bacc_cell = ''
                    auc_cell = ''
                
                row_parts.extend([bacc_cell, auc_cell])
            
            model_rows.append(' & '.join(row_parts) + ' \\\\')
        
        # Add multirow for first row
        if model_rows:
            model_rows[0] = f"     \\multirow{{4}}{{*}}{{{model_type}}}" + model_rows[0][len(f"     "):]
            rows.extend(model_rows)
            rows.append('[3pt]')
    
    return '\n'.join(rows[:-1])  # Remove last [3pt]

# Run the collection
print("="*70)
print("Collecting results from experiment directories...")
print("="*70)

all_results = collect_all_results()

print("\n" + "="*70)
print("DEBUG: Results summary")
print("="*70)
for dataset in sorted(all_results.keys()):
    print(f"\n{dataset}:")
    for model in sorted(all_results[dataset].keys()):
        setups = list(all_results[dataset][model].keys())
        print(f"  {model}: {setups}")

print("\n" + "="*70)
print("Generating LaTeX tables...")
print("="*70)

# Generate Table 1
print("\n" + "="*70)
print("TABLE 1: Path, Derma-e, Derma-e ext., OCT, Pneum.")
print("="*70)
table1_rows = generate_latex_table_rows(all_results, DATASETS_TABLE1)
print(table1_rows)

# Generate Table 2
print("\n" + "="*70)
print("TABLE 2: Breast, Blood, Tissue, Organ, AMOS")
print("="*70)
table2_rows = generate_latex_table_rows(all_results, DATASETS_TABLE2)
print(table2_rows)

print("\n" + "="*70)
print("✓ Table generation complete!")
print("="*70)

# Evaluate OrganaMNIST Models on AMOS External Test Set

Evaluate the trained OrganaMNIST models on the AMOS-2022 external test set to generate ensemble and per-fold metrics.

In [ ]:
import sys
import torch
import torch.nn as nn
from pathlib import Path
import numpy as np
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize
import json
import re

# Add parent directories to path
sys.path.insert(0, str(Path('../utils').resolve()))

from utils.train_models_load_datasets import load_models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# AMOS-2022 to OrganaMNIST class mapping (6 organs present in both datasets)
AMOS_TO_ORGANAMNIST = {
    0: 10,  # spleen → spleen
    1: 5,   # right kidney → kidney-right
    2: 4,   # left kidney → kidney-left
    5: 6,   # liver → liver (note: AMOS 3=gall bladder, 4=esophagus, 6=stomach are skipped)
    9: 9,   # pancreas → pancreas
    13: 0,  # bladder → bladder (URINARY bladder, not gall bladder!)
}

# Minimal parse_experiment_config for this cell
EXP_DIR_PATTERN = re.compile(
    r'(?P<model>resnet18|vit_b_16)_224_.*?randaug(?P<randaug>[01]).*?(?:dropout(?P<dropout>\d\.\d+))?$',
)
def parse_experiment_config(exp_name):
    match = EXP_DIR_PATTERN.search(exp_name)
    if not match:
        return None
    model = 'resnet18' if 'resnet18' in match.group('model') else 'vit_b_16'
    has_aug = match.group('randaug') == '1'
    has_dropout = match.group('dropout') is not None
    if has_aug and has_dropout:
        setup = 'DADO'
    elif has_aug:
        setup = 'DA'
    elif has_dropout:
        setup = 'DO'
    else:
        setup = '--'
    return {
        'model': model,
        'setup': setup,
        'has_aug': has_aug,
        'has_dropout': has_dropout,
        'dropout_value': match.group('dropout')
    }

def load_and_prepare_amos_data(data_path='../Data/AMOS_2022/amos_external_test_224.npz'):
    """Load AMOS external test data and map labels to OrganaMNIST."""
    data = np.load(data_path)
    images = data['test_images']  # Shape: (N, 224, 224, 1)
    amos_labels = data['test_labels']  # Shape: (N, 15) one-hot AMOS labels
    
    # Convert one-hot to class indices
    amos_class_ids = np.argmax(amos_labels, axis=1)
    
    # Filter: keep only samples with organs that exist in OrganaMNIST
    mapped_indices = []
    organamnist_labels = []
    
    for idx, amos_id in enumerate(amos_class_ids):
        if amos_id in AMOS_TO_ORGANAMNIST:
            mapped_indices.append(idx)
            organamnist_labels.append(AMOS_TO_ORGANAMNIST[amos_id])
    
    mapped_indices = np.array(mapped_indices)
    filtered_images = images[mapped_indices]
    filtered_labels = np.array(organamnist_labels)
    
    # Convert to PyTorch tensors (normalize and repeat to 3 channels)
    images_tensor = torch.from_numpy(filtered_images).float() / 255.0
    images_tensor = images_tensor.permute(0, 3, 1, 2)  # (N, 1, 224, 224)
    images_tensor = images_tensor.repeat(1, 3, 1, 1)    # (N, 3, 224, 224) for ResNet
    
    # Normalize (match training preprocessing)
    mean = torch.tensor([0.5, 0.5, 0.5]).view(1, 3, 1, 1)
    std = torch.tensor([0.5, 0.5, 0.5]).view(1, 3, 1, 1)
    images_tensor = (images_tensor - mean) / std
    
    labels_tensor = torch.from_numpy(filtered_labels).long()
    
    print(f"Filtered AMOS dataset:")
    print(f"  Original samples: {len(images)}")
    print(f"  Mapped samples (ID-compatible): {len(filtered_images)}")
    print(f"  Unique classes: {np.unique(filtered_labels)}")
    
    return images_tensor, labels_tensor

def evaluate_models_on_amos(experiments, images, labels, batch_size=64):
    """Evaluate OrganaMNIST models on AMOS test set."""
    results = {}
    
    dataset = torch.utils.data.TensorDataset(images, labels)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    # Get unique classes present in AMOS for AUC computation
    unique_classes = torch.unique(labels).numpy()
    
    for exp_name, exp_data in experiments.items():
        print(f"\nEvaluating {exp_name}...")
        models = exp_data['models']
        
        all_fold_probs = []
        all_fold_preds = []
        
        with torch.no_grad():
            for model in models:
                fold_probs_list = []
                fold_preds_list = []
                
                for batch_imgs, _ in loader:
                    batch_imgs = batch_imgs.to(device)
                    outputs = model(batch_imgs)
                    probs = torch.softmax(outputs, dim=1)
                    preds = torch.argmax(probs, dim=1)
                    
                    fold_probs_list.append(probs.cpu().numpy())
                    fold_preds_list.append(preds.cpu().numpy())
                
                fold_probs = np.concatenate(fold_probs_list, axis=0)
                fold_preds = np.concatenate(fold_preds_list, axis=0)
                
                all_fold_probs.append(fold_probs)
                all_fold_preds.append(fold_preds)
        
        all_fold_probs = np.array(all_fold_probs)  # (5, N, 11)
        all_fold_preds = np.array(all_fold_preds)  # (5, N)
        
        # Ensemble prediction (average probabilities across folds)
        ensemble_probs = np.mean(all_fold_probs, axis=0)  # (N, 11)
        ensemble_preds = np.argmax(ensemble_probs, axis=1)  # (N,)
        
        labels_np = labels.numpy()
        
        # Compute ensemble metrics
        ensemble_bacc = balanced_accuracy_score(labels_np, ensemble_preds)
        
        # AUC: binarize labels for only classes present in AMOS
        y_true_bin = label_binarize(labels_np, classes=unique_classes)
        y_score_filtered = ensemble_probs[:, unique_classes]
        ensemble_auc = roc_auc_score(y_true_bin, y_score_filtered, average='macro')
        
        # Per-fold metrics
        fold_baccs = []
        fold_aucs = []
        
        for fold_idx in range(len(models)):
            fold_bacc = balanced_accuracy_score(labels_np, all_fold_preds[fold_idx])
            fold_baccs.append(fold_bacc)
            
            y_score_fold_filtered = all_fold_probs[fold_idx][:, unique_classes]
            fold_auc = roc_auc_score(y_true_bin, y_score_fold_filtered, average='macro')
            fold_aucs.append(fold_auc)
        
        # Std across folds
        std_bacc = np.std(fold_baccs, ddof=1)
        std_auc = np.std(fold_aucs, ddof=1)
        
        results[exp_name] = {
            'ensemble_bacc': ensemble_bacc,
            'ensemble_auc': ensemble_auc,
            'std_bacc': std_bacc,
            'std_auc': std_auc,
            'fold_baccs': fold_baccs,
            'fold_aucs': fold_aucs
        }
        
        print(f"  Ensemble: bAcc={ensemble_bacc:.3f}±{std_bacc:.3f}, AUC={ensemble_auc:.3f}±{std_auc:.3f}")
    
    return results

def load_organamnist_experiments():
    """Load all OrganaMNIST experiment models using load_models()."""
    organamnist_dir = Path('organamnist')
    
    if not organamnist_dir.exists():
        print(f"Error: {organamnist_dir} not found!")
        return {}
    
    experiments = {}
    
    for exp_dir in organamnist_dir.iterdir():
        if not exp_dir.is_dir():
            continue
        
        config = parse_experiment_config(exp_dir.name)
        if not config:
            continue
        
        model_type = config['model']
        setup = config['setup']
        
        try:
            models = load_models(
                flag='organamnist',
                device=device,
                size=224,
                model_backbone=model_type,
                setup=setup
            )
            
            if len(models) == 5:
                for model in models:
                    model.eval()
                
                experiments[f"{model_type}_{setup}"] = {
                    'models': models,
                    'config': config
                }
                print(f"✅ Loaded {model_type} {setup}: 5 fold models")
            else:
                print(f"⚠️  Only {len(models)}/5 models for {model_type} {setup}")
        except Exception as e:
            print(f"❌ Error loading {exp_dir.name}: {e}")
    
    return experiments

# Main evaluation
print("="*70)
print("AMOS-2022 External Test Evaluation")
print("="*70)

# Load AMOS data
print("\nLoading AMOS data...")
amos_images, amos_labels = load_and_prepare_amos_data()
print(f"Loaded {len(amos_images)} images")

# Load OrganaMNIST models
print("\nLoading OrganaMNIST models...")
experiments = load_organamnist_experiments()

if experiments:
    print("\n" + "="*70)
    print("Running evaluation...")
    print("="*70)
    amos_results = evaluate_models_on_amos(experiments, amos_images, amos_labels)
    
    print("\n" + "="*70)
    print("AMOS-2022 Results Summary")
    print("="*70)
    for exp_name, metrics in sorted(amos_results.items()):
        print(f"\n{exp_name}:")
        print(f"  Ensemble b-Acc: {metrics['ensemble_bacc']:.4f} ± {metrics['std_bacc']:.4f}")
        print(f"  Ensemble AUC:   {metrics['ensemble_auc']:.4f} ± {metrics['std_auc']:.4f}")
else:
    print("❌ No experiments found!")

# Evaluate DermaMNIST-e Models on External Test Subset

Evaluate the trained DermaMNIST-e models on the external site subset of the test set (samples where center contains "external").

In [ ]:
import sys
import torch
import torch.nn as nn
from pathlib import Path
import numpy as np
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from torch.utils.data import DataLoader
from torchvision import transforms
import re

# Add parent directories to path
sys.path.insert(0, str(Path('../utils').resolve()))

from utils.train_models_load_datasets import load_models, load_datasets

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Parse experiment configuration (reuse from above)
EXP_DIR_PATTERN_DERMA = re.compile(
    r'(?P<model>resnet18|vit_b_16)_224_.*?randaug(?P<randaug>[01]).*?(?:dropout(?P<dropout>\d\.\d+))?$',
)
def parse_experiment_config_derma(exp_name):
    match = EXP_DIR_PATTERN_DERMA.search(exp_name)
    if not match:
        return None
    model = 'resnet18' if 'resnet18' in match.group('model') else 'vit_b_16'
    has_aug = match.group('randaug') == '1'
    has_dropout = match.group('dropout') is not None
    if has_aug and has_dropout:
        setup = 'DADO'
    elif has_aug:
        setup = 'DA'
    elif has_dropout:
        setup = 'DO'
    else:
        setup = '--'
    return {
        'model': model,
        'setup': setup,
        'has_aug': has_aug,
        'has_dropout': has_dropout,
        'dropout_value': match.group('dropout')
    }

def load_dermamnist_test_data():
    """Load DermaMNIST-e test dataset."""
    flag = 'dermamnist-e'
    color = True
    size = 224
    batch_size = 128
    
    transform_eval = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[.5, .5, .5], std=[.5, .5, .5])
    ])
    
    # Load test dataset
    [_, _, test_dataset], [_, _, _], info = load_datasets(
        flag, color, size, transform_eval, batch_size, cache_test=False
    )
    
    return test_dataset, batch_size

def filter_external_samples(test_dataset):
    """Filter test samples from external sites."""
    external_indices = [
        i for i, c in enumerate(test_dataset.test_centers) 
        if isinstance(c, str) and ("external" in c.lower())
    ]
    return external_indices

def evaluate_dermamnist_external(experiments, test_dataset, batch_size=128):
    """Evaluate DermaMNIST-e models on external test subset."""
    results = {}
    
    # Filter external samples
    external_indices = filter_external_samples(test_dataset)
    external_subset = torch.utils.data.Subset(test_dataset, external_indices)
    external_loader = DataLoader(external_subset, batch_size=batch_size, shuffle=False)
    
    print(f"External test samples: {len(external_indices)}")
    
    for exp_name, exp_data in experiments.items():
        print(f"\nEvaluating {exp_name} on external subset...")
        models = exp_data['models']
        
        # Set models to eval mode
        for m in models:
            m.eval()
        
        y_true_external = []
        y_pred_external = []
        y_prob_external = []
        
        with torch.no_grad():
            # Collect per-model outputs
            per_model_probs = [[] for _ in models]
            per_model_preds = [[] for _ in models]
            
            for x, y in external_loader:
                x = x.to(device, non_blocking=True)
                
                probs_list = []
                for mi, m in enumerate(models):
                    logits = m(x)
                    probs = torch.softmax(logits, dim=1)
                    probs_list.append(probs)
                    
                    # Save per-model outputs on CPU
                    per_model_probs[mi].append(probs.cpu())
                    per_model_preds[mi].append(probs.argmax(dim=1).cpu())
                
                # Ensemble prediction (average probabilities)
                probs_ens = torch.stack(probs_list, dim=0).mean(dim=0)  # [B, C]
                preds = probs_ens.argmax(dim=1)  # [B]
                
                y_prob_external.append(probs_ens.cpu())
                y_pred_external.append(preds.cpu())
                y_true_external.append(y.view(-1).cpu().long())
        
        y_prob_external = torch.cat(y_prob_external, dim=0)
        y_pred_external = torch.cat(y_pred_external, dim=0)
        y_true_external = torch.cat(y_true_external, dim=0)
        
        # Compute ensemble metrics
        acc_external = (y_pred_external == y_true_external).float().mean().item()
        bacc_external = balanced_accuracy_score(
            y_true_external.numpy(), y_pred_external.numpy()
        )
        
        num_classes = y_prob_external.shape[1]
        try:
            auc_external = roc_auc_score(
                y_true_external.numpy(),
                y_prob_external.numpy(),
                labels=list(range(num_classes)),
                multi_class='ovr',
                average='macro',
            )
        except Exception:
            auc_external = float('nan')
        
        # Compute std of metrics over individual models
        accs_external = []
        baccs_external = []
        aucs_external = []
        
        for mi in range(len(models)):
            probs_i = torch.cat(per_model_probs[mi], dim=0)  # [N, C]
            preds_i = torch.cat(per_model_preds[mi], dim=0)  # [N]
            
            acc_i = (preds_i == y_true_external).float().mean().item()
            bacc_i = balanced_accuracy_score(y_true_external.numpy(), preds_i.numpy())
            try:
                auc_i = roc_auc_score(
                    y_true_external.numpy(),
                    probs_i.numpy(),
                    labels=list(range(num_classes)),
                    multi_class='ovr',
                    average='macro',
                )
            except Exception:
                auc_i = float('nan')
            
            accs_external.append(acc_i)
            baccs_external.append(bacc_i)
            aucs_external.append(auc_i)
        
        # Compute std (ddof=1 for sample std, consistent with AMOS evaluation)
        std_bacc = np.std(baccs_external, ddof=1)
        std_auc = np.std([a for a in aucs_external if not np.isnan(a)], ddof=1) if any(not np.isnan(a) for a in aucs_external) else float('nan')
        
        results[exp_name] = {
            'ensemble_acc': acc_external,
            'ensemble_bacc': bacc_external,
            'ensemble_auc': auc_external,
            'std_bacc': std_bacc,
            'std_auc': std_auc,
            'fold_baccs': baccs_external,
            'fold_aucs': aucs_external
        }
        
        print(f"  Ensemble: bAcc={bacc_external:.3f}±{std_bacc:.3f}, AUC={auc_external:.3f}±{std_auc:.3f}")
    
    return results

def load_dermamnist_experiments():
    """Load all DermaMNIST-e experiment models."""
    dermamnist_dir = Path('dermamnist-e')
    
    if not dermamnist_dir.exists():
        print(f"Error: {dermamnist_dir} not found!")
        return {}
    
    experiments = {}
    
    for exp_dir in dermamnist_dir.iterdir():
        if not exp_dir.is_dir():
            continue
        
        config = parse_experiment_config_derma(exp_dir.name)
        if not config:
            continue
        
        model_type = config['model']
        setup = config['setup']
        
        try:
            models = load_models(
                flag='dermamnist-e',
                device=device,
                size=224,
                model_backbone=model_type,
                setup=setup
            )
            
            if len(models) == 5:
                experiments[f"{model_type}_{setup}"] = {
                    'models': models,
                    'config': config
                }
                print(f"✅ Loaded {model_type} {setup}: 5 fold models")
            else:
                print(f"⚠️  Only {len(models)}/5 models for {model_type} {setup}")
        except Exception as e:
            print(f"❌ Error loading {exp_dir.name}: {e}")
    
    return experiments

# Main evaluation
print("="*70)
print("DermaMNIST-e External Subset Evaluation")
print("="*70)

# Load test dataset
print("\nLoading DermaMNIST-e test dataset...")
test_dataset, batch_size = load_dermamnist_test_data()
print(f"Total test samples: {len(test_dataset)}")

# Load DermaMNIST-e models
print("\nLoading DermaMNIST-e models...")
experiments = load_dermamnist_experiments()

if experiments:
    print("\n" + "="*70)
    print("Running evaluation on external subset...")
    print("="*70)
    derma_external_results = evaluate_dermamnist_external(experiments, test_dataset, batch_size)
    
    print("\n" + "="*70)
    print("DermaMNIST-e External Subset Results Summary")
    print("="*70)
    for exp_name, metrics in sorted(derma_external_results.items()):
        print(f"\n{exp_name}:")
        print(f"  Ensemble b-Acc: {metrics['ensemble_bacc']:.4f} ± {metrics['std_bacc']:.4f}")
        print(f"  Ensemble AUC:   {metrics['ensemble_auc']:.4f} ± {metrics['std_auc']:.4f}")
else:
    print("❌ No experiments found!")

# Verify: Evaluate DermaMNIST-e Models on Complete Test Set

Compare complete test set vs external subset performance to verify that complete test set has expected balanced accuracy (0.657 for ResNet DADO).

In [ ]:
# Load ResNet18 DADO model specifically for verification
print("="*70)
print("Loading ResNet18 DADO model for verification...")
print("="*70)

dado_models = load_models(
    flag='dermamnist-e',
    device=device,
    size=224,
    model_backbone='resnet18',
    setup='DADO'
)

print(f"Loaded {len(dado_models)} DADO models")

# Evaluate on complete test set
complete_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

for m in dado_models:
    m.eval()

y_true_complete = []
y_pred_complete = []
y_prob_complete = []

print(f"\nEvaluating on complete test set ({len(test_dataset)} samples)...")

with torch.no_grad():
    # Collect per-model outputs
    per_model_probs = [[] for _ in dado_models]
    per_model_preds = [[] for _ in dado_models]
    
    for x, y in complete_loader:
        x = x.to(device, non_blocking=True)
        
        probs_list = []
        for mi, m in enumerate(dado_models):
            logits = m(x)
            probs = torch.softmax(logits, dim=1)
            probs_list.append(probs)
            
            # Save per-model outputs on CPU
            per_model_probs[mi].append(probs.cpu())
            per_model_preds[mi].append(probs.argmax(dim=1).cpu())
        
        # Ensemble prediction (average probabilities)
        probs_ens = torch.stack(probs_list, dim=0).mean(dim=0)  # [B, C]
        preds = probs_ens.argmax(dim=1)  # [B]
        
        y_prob_complete.append(probs_ens.cpu())
        y_pred_complete.append(preds.cpu())
        y_true_complete.append(y.view(-1).cpu().long())

y_prob_complete = torch.cat(y_prob_complete, dim=0)
y_pred_complete = torch.cat(y_pred_complete, dim=0)
y_true_complete = torch.cat(y_true_complete, dim=0)

# Compute ensemble metrics
acc_complete = (y_pred_complete == y_true_complete).float().mean().item()
bacc_complete = balanced_accuracy_score(
    y_true_complete.numpy(), y_pred_complete.numpy()
)

num_classes = y_prob_complete.shape[1]
try:
    auc_complete = roc_auc_score(
        y_true_complete.numpy(),
        y_prob_complete.numpy(),
        labels=list(range(num_classes)),
        multi_class='ovr',
        average='macro',
    )
except Exception:
    auc_complete = float('nan')

# Compute std of metrics over individual models
baccs_complete = []
aucs_complete = []

for mi in range(len(dado_models)):
    probs_i = torch.cat(per_model_probs[mi], dim=0)  # [N, C]
    preds_i = torch.cat(per_model_preds[mi], dim=0)  # [N]
    
    bacc_i = balanced_accuracy_score(y_true_complete.numpy(), preds_i.numpy())
    try:
        auc_i = roc_auc_score(
            y_true_complete.numpy(),
            probs_i.numpy(),
            labels=list(range(num_classes)),
            multi_class='ovr',
            average='macro',
        )
    except Exception:
        auc_i = float('nan')
    
    baccs_complete.append(bacc_i)
    aucs_complete.append(auc_i)

std_bacc_complete = np.std(baccs_complete, ddof=1)
std_auc_complete = np.std([a for a in aucs_complete if not np.isnan(a)], ddof=1) if any(not np.isnan(a) for a in aucs_complete) else float('nan')

print("\n" + "="*70)
print("ResNet18 DADO - Complete Test Set Results")
print("="*70)
print(f"Ensemble b-Acc: {bacc_complete:.4f} ± {std_bacc_complete:.4f}")
print(f"Ensemble AUC:   {auc_complete:.4f} ± {std_auc_complete:.4f}")
print(f"Expected b-Acc: 0.6570")

# Compare with external subset
external_bacc = derma_external_results['resnet18_DADO']['ensemble_bacc']
external_std = derma_external_results['resnet18_DADO']['std_bacc']

print("\n" + "="*70)
print("COMPARISON: Complete vs External Subset (ResNet18 DADO)")
print("="*70)
print(f"Complete test set b-Acc: {bacc_complete:.4f} ± {std_bacc_complete:.4f}")
print(f"External subset b-Acc:   {external_bacc:.4f} ± {external_std:.4f}")
print(f"Difference:              {(external_bacc - bacc_complete):+.4f} {'⚠️ EXTERNAL > COMPLETE' if external_bacc > bacc_complete else '✓'}")

# Visualize Results: Box Plots for All Datasets

Generate box plots showing balanced accuracy results for each dataset, comparing ResNet18 and ViT models across all setups (--, DA, DO, DADO). Derma-e external results are overlaid with transparency.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import json
import numpy as np

# Dataset names for display
DATASET_DISPLAY_NAMES = {
    'pathmnist': 'Path',
    'dermamnist-e': 'Derma-e',
    'octmnist': 'OCT',
    'pneumoniamnist': 'Pneumonia',
    'breastmnist': 'Breast',
    'bloodmnist': 'Blood',
    'tissuemnist': 'Tissue',
    'organamnist': 'Organ'
}

# Setup colors and markers
SETUP_STYLES = {
    '--': {'color': '#1f77b4', 'marker': 'o', 'label': '--'},
    'DA': {'color': '#ff7f0e', 'marker': 's', 'label': 'DA'},
    'DO': {'color': '#2ca02c', 'marker': '^', 'label': 'DO'},
    'DADO': {'color': '#d62728', 'marker': 'D', 'label': 'DADO'}
}

def load_fold_metrics_from_dir(exp_dir):
    """Load per-fold balanced accuracy from experiment directory."""
    exp_path = Path(exp_dir)
    fold_baccs = []
    
    # Try both naming patterns: metrics_fold_#_test.json (ResNet) and metrics_fold_#.json (ViT)
    fold_files = sorted(exp_path.glob('metrics_fold_*_test.json'))
    if not fold_files:
        # Fallback to ViT pattern, but exclude ensemble file
        fold_files = sorted([f for f in exp_path.glob('metrics_fold_*.json') 
                            if f.name != 'metrics_ensemble.json'])
    
    for fold_file in fold_files:
        try:
            with open(fold_file, 'r') as f:
                fold_data = json.load(f)
                metrics = fold_data.get('metrics', {})
                bacc = metrics.get('balanced_accuracy', metrics.get('balanced_acc', np.nan))
                if not np.isnan(bacc):
                    fold_baccs.append(bacc)
        except Exception:
            continue
    
    return fold_baccs if fold_baccs else None

def load_ensemble_metric_from_dir(exp_dir):
    """Load ensemble balanced accuracy from metrics_ensemble.json."""
    exp_path = Path(exp_dir)
    ensemble_file = exp_path / 'metrics_ensemble.json'
    
    if ensemble_file.exists():
        try:
            with open(ensemble_file, 'r') as f:
                ensemble_data = json.load(f)
                metrics = ensemble_data.get('metrics', {})
                bacc = metrics.get('balanced_accuracy', metrics.get('balanced_acc', np.nan))
                if not np.isnan(bacc):
                    return bacc
        except Exception:
            pass
    
    return None

def parse_experiment_config(exp_name):
    """Parse experiment directory name to extract model and setup.
    
    Format: backbone_224_timestamp_randaug[0|1]_numepochs100_bs128[_dropout0.X]
    - randaug0 + no dropout = '--'
    - randaug1 + no dropout = 'DA'
    - randaug0 + dropout = 'DO'
    - randaug1 + dropout = 'DADO'
    """
    if 'resnet18' in exp_name:
        model = 'resnet18'
    elif 'vit_b_16' in exp_name:
        model = 'vit_b_16'
    else:
        return None
    
    has_randaug = 'randaug1' in exp_name
    has_dropout = 'dropout' in exp_name
    
    if not has_randaug and not has_dropout:
        setup = '--'
    elif has_randaug and not has_dropout:
        setup = 'DA'
    elif not has_randaug and has_dropout:
        setup = 'DO'
    else:  # has_randaug and has_dropout
        setup = 'DADO'
    
    return {'model': model, 'setup': setup}

def collect_boxplot_data():
    """Collect fold-level balanced accuracy data for box plots."""
    data = {}
    
    runs_dir = Path('.')
    
    for dataset_dir in runs_dir.iterdir():
        if not dataset_dir.is_dir():
            continue
        
        dataset_name = dataset_dir.name.lower()
        
        # Skip non-dataset directories
        if dataset_name not in DATASET_DISPLAY_NAMES:
            continue
        
        data[dataset_name] = {'resnet18': {}, 'vit_b_16': {}}
        
        # Find all experiment directories
        for exp_dir in dataset_dir.iterdir():
            if not exp_dir.is_dir():
                continue
            
            exp_name = exp_dir.name
            
            # Parse experiment configuration
            config = parse_experiment_config(exp_name)
            if not config:
                continue
            
            model = config['model']
            setup = config['setup']
            
            # Load fold metrics and ensemble metric
            fold_baccs = load_fold_metrics_from_dir(exp_dir)
            ensemble_bacc = load_ensemble_metric_from_dir(exp_dir)
            
            if fold_baccs and len(fold_baccs) == 5 and ensemble_bacc is not None:
                data[dataset_name][model][setup] = {
                    'fold_baccs': fold_baccs,
                    'ensemble_bacc': ensemble_bacc
                }
    
    return data

def create_boxplot_for_dataset(ax, dataset_name, dataset_data, derma_external_data=None):
    """Create box plot for a single dataset."""
    models = ['resnet18', 'vit_b_16']
    setups = ['--', 'DA', 'DO', 'DADO']
    
    positions = {
        'resnet18': {
            '--': 1,
            'DA': 2,
            'DO': 3,
            'DADO': 4
        },
        'vit_b_16': {
            '--': 6,
            'DA': 7,
            'DO': 8,
            'DADO': 9
        }
    }
    
    all_boxes_data = []
    all_positions = []
    
    # Collect data for box plots
    for model in models:
        for setup in setups:
            if setup in dataset_data.get(model, {}):
                fold_values = dataset_data[model][setup]['fold_baccs']
                all_boxes_data.append(fold_values)
                all_positions.append(positions[model][setup])
    
    # Create box plots
    if all_boxes_data:
        bp = ax.boxplot(all_boxes_data, positions=all_positions, widths=0.6,
                        patch_artist=True, showfliers=False,
                        boxprops=dict(facecolor='lightgray', alpha=0.3, linewidth=1.5),
                        medianprops=dict(color='black', linewidth=2),
                        whiskerprops=dict(linewidth=1.5),
                        capprops=dict(linewidth=1.5))
    
    # Overlay ensemble points
    for model in models:
        for setup in setups:
            if setup in dataset_data.get(model, {}):
                ensemble_bacc = dataset_data[model][setup]['ensemble_bacc']
                pos = positions[model][setup]
                
                style = SETUP_STYLES[setup]
                ax.plot(pos, ensemble_bacc, marker=style['marker'], 
                       color=style['color'], markersize=10, 
                       markeredgecolor='black', markeredgewidth=1.5, zorder=3)
    
    # Overlay external results if available
    if derma_external_data:
        if dataset_name == 'dermamnist-e':
            # Derma-e external results
            for model in models:
                for setup in setups:
                    key = f'{model}_{setup}'
                    if key in derma_external_data:
                        external_bacc = derma_external_data[key]['ensemble_bacc']
                        fold_baccs = derma_external_data[key]['fold_baccs']
                        
                        # Offset position slightly to the right
                        pos = positions[model][setup] + 0.3
                        
                        style = SETUP_STYLES[setup]
                        # Plot with transparency
                        ax.plot(pos, external_bacc, marker=style['marker'], 
                               color=style['color'], markersize=10, alpha=0.5,
                               markeredgecolor='black', markeredgewidth=1.5, zorder=3)
                        
                        # Add small box plot for external data
                        bp_ext = ax.boxplot([fold_baccs], positions=[pos], widths=0.3,
                                           patch_artist=True, showfliers=False,
                                           boxprops=dict(facecolor=style['color'], alpha=0.15, linewidth=1),
                                           medianprops=dict(color='black', linewidth=1.5, alpha=0.5),
                                           whiskerprops=dict(linewidth=1, alpha=0.5),
                                           capprops=dict(linewidth=1, alpha=0.5))
        
        elif dataset_name == 'organamnist':
            # AMOS external results overlaid on OrganaMNIST
            try:
                if 'amos_results' in globals() and amos_results is not None:
                    for model in models:
                        for setup in setups:
                            key = f'{model}_{setup}'
                            if key in amos_results:
                                amos_bacc = amos_results[key]['ensemble_bacc']
                                amos_fold_baccs = amos_results[key]['fold_baccs']
                                
                                # Offset position slightly to the right
                                pos = positions[model][setup] + 0.3
                                
                                style = SETUP_STYLES[setup]
                                # Plot with transparency
                                ax.plot(pos, amos_bacc, marker=style['marker'], 
                                       color=style['color'], markersize=10, alpha=0.5,
                                       markeredgecolor='black', markeredgewidth=1.5, zorder=3)
                                
                                # Add small box plot for AMOS data
                                bp_ext = ax.boxplot([amos_fold_baccs], positions=[pos], widths=0.3,
                                                   patch_artist=True, showfliers=False,
                                                   boxprops=dict(facecolor=style['color'], alpha=0.15, linewidth=1),
                                                   medianprops=dict(color='black', linewidth=1.5, alpha=0.5),
                                                   whiskerprops=dict(linewidth=1, alpha=0.5),
                                                   capprops=dict(linewidth=1, alpha=0.5))
            except NameError:
                pass
    
    # Formatting
    ax.set_xticks([2.5, 7.5])
    ax.set_xticklabels(['ResNet18', 'ViT-B/16'], fontsize=11, fontweight='bold')
    ax.set_ylabel('Balanced Accuracy', fontsize=11)
    ax.set_title(DATASET_DISPLAY_NAMES.get(dataset_name, dataset_name), 
                fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_xlim(0, 10)
    
    # Set y-axis limits based on data with increased padding
    if all_boxes_data:
        all_values = [v for data in all_boxes_data for v in data]
        
        # Also include ensemble values from main dataset
        for model in models:
            for setup in setups:
                if setup in dataset_data.get(model, {}):
                    all_values.append(dataset_data[model][setup]['ensemble_bacc'])
        
        # Include external data in y-axis calculation if available
        if dataset_name == 'dermamnist-e' and derma_external_data is not None:
            for model in models:
                for setup in setups:
                    key = f'{model}_{setup}'
                    if key in derma_external_data:
                        all_values.extend(derma_external_data[key]['fold_baccs'])
                        all_values.append(derma_external_data[key]['ensemble_bacc'])
        
        if dataset_name == 'organamnist':
            try:
                # Check if amos_results exists in the global namespace
                if 'amos_results' in globals() and amos_results is not None:
                    for model in models:
                        for setup in setups:
                            key = f'{model}_{setup}'
                            if key in amos_results:
                                all_values.extend(amos_results[key]['fold_baccs'])
                                all_values.append(amos_results[key]['ensemble_bacc'])
            except NameError:
                pass
        
        y_min, y_max = min(all_values), max(all_values)
        y_range = y_max - y_min
        ax.set_ylim(y_min - 0.15 * y_range, y_max + 0.15 * y_range)

# Collect all data
print("Collecting fold-level data for visualization...")
boxplot_data = collect_boxplot_data()

# Debug: Print what was collected
print("\nData collected:")
for dataset_name, models_data in boxplot_data.items():
    print(f"  {dataset_name}:")
    for model, setups_data in models_data.items():
        print(f"    {model}: {list(setups_data.keys())}")

# Add external data for Derma-e and AMOS
derma_ext_data = derma_external_results if 'derma_external_results' in dir() else None
print(f"\nDerma-e external data available: {derma_ext_data is not None}")
print(f"AMOS data available: {'amos_results' in dir()}")

# Create figure with subplots
datasets_to_plot = ['pathmnist', 'dermamnist-e', 'octmnist', 'pneumoniamnist',
                    'breastmnist', 'bloodmnist', 'tissuemnist', 'organamnist']

n_datasets = len(datasets_to_plot)
n_cols = 4
n_rows = (n_datasets + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5*n_rows))
axes = axes.flatten()

# Create plots
for idx, dataset_name in enumerate(datasets_to_plot):
    if dataset_name in boxplot_data:
        create_boxplot_for_dataset(axes[idx], dataset_name, boxplot_data[dataset_name], derma_ext_data)
    else:
        axes[idx].text(0.5, 0.5, f'No data for {dataset_name}', 
                      ha='center', va='center', fontsize=12)
        axes[idx].set_xlim(0, 10)
        axes[idx].set_xticks([])

# Hide unused subplots
for idx in range(n_datasets, len(axes)):
    axes[idx].axis('off')

# Create legend
legend_elements = [
    mpatches.Patch(facecolor='lightgray', edgecolor='black', alpha=0.3, label='Fold variability (box plot)')
]
for setup, style in SETUP_STYLES.items():
    legend_elements.append(
        plt.Line2D([0], [0], marker=style['marker'], color='w', 
                  markerfacecolor=style['color'], markeredgecolor='black',
                  markersize=10, label=f"{style['label']} (ensemble)")
    )

# Check if external data is available
has_external = (derma_ext_data is not None) or ('amos_results' in globals() and amos_results is not None)
if has_external:
    legend_elements.append(
        plt.Line2D([0], [0], marker='o', color='w', 
                  markerfacecolor='gray', markeredgecolor='black',
                  markersize=10, alpha=0.5, label='External test (Derma-e, AMOS)')
    )

fig.legend(handles=legend_elements, loc='upper center', ncol=6, 
          fontsize=11, frameon=True, bbox_to_anchor=(0.5, 1.02))

plt.tight_layout(rect=[0, 0, 1, 0.98])

# Save figure (create directory if needed)
output_path = Path('../../../uq_benchmark_results/figures')
output_path.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path / 'all_datasets_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Box plots generated and saved to {output_path / 'all_datasets_boxplots.png'}")